# 04 — Make retrieval useful before adding a model

Agentic systems need retrieval, but retrieval does not have to begin with an
embedding service. BM25 gives OSII a dependable local baseline for exact
terminology, identifiers, names, and phrases. It works offline and provides a
reference point for evaluating more complex methods.

Most importantly, the index is not canonical. It can be deleted, tuned, and
rebuilt from preferred text while object identity and provenance stay stable.

In [1]:
from osii.domain.scopes.collections import list_collections
from osii.domain.services.search import dashboard_search
from osii.search.lexical import build_bm25_index

from _demo_support import demo_paths, require_path

/Users/heidi/Projects/OSII/osii/osii-demo-notebooks/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
                                                                                                                       


Index files
-----------
BM25 index: /Users/heidi/Projects/OSII/osii/osii-demo-notebooks/demo-workspace/.osii/embeddings/lexical/bm25_index.pkl
Metadata: /Users/heidi/Projects/OSII/osii/osii-demo-notebooks/demo-workspace/.osii/embeddings/lexical/bm25_meta.json


In [ ]:
paths = demo_paths()
require_path(paths.osii_root / "objects", "Run the extraction example first.")

## Build a derived index

Chunking creates retrieval-sized views over preferred object text. Those
chunks keep character offsets back into the object, so a good match can return
to evidence rather than becoming an orphaned string in a vector database.

In [2]:
index_path, metadata_path = build_bm25_index(paths.osii_root)

print("BM25 index:", index_path)
print("Chunk metadata:", metadata_path)


Root results (lexical)
----------------------
- purcell.pdf  score=63.000
  The demonstration involved a tall rectangular transparent vessel of corn syrup, projected by an overhead projector turned on its side. Some essential hand waving could not be reproduced.

This is a talk that I would not, I’m afraid, have the nerve to give under any other circumstances, It’s a story I’ve been saving up to tell Viki. Like so many of you here, I’ve enjoyed from time to time the wonde
  chars=550:1279
- purcell.pdf  score=42.000
  We wander around strictly as amateurs equipped only with some elementary physics, and in the end, it turns out, we improve our understanding of the elemen- tary physics even if we don’t throw much light on the other subjects. Now this is that kind of a subject, but I have still another reason for wanting to, as it were, needle Viki with it, because I’m going to talk for a while about viscosity. Vi
  chars=1054:1808
- purcell.pdf  score=10.000
  The viscosity of a liquid 

## State a research question as an ordinary query

Keeping the query in its own cell makes adaptation obvious. Replace it with
the language used in your own corpus and compare results before changing the
retrieval algorithm.

In [3]:
ROOT_QUERY = "low Reynolds number viscosity swimming microorganisms"

print(ROOT_QUERY)


Collection-scoped results
-------------------------
- purcell.pdf -> Life at low Reynolds number

E. M. Purcell Lyman Laboratory, Harvard University, Cambridge, Massachusetts 02138 (Received 12 June 1976)

Editor’s note: This is a reprint (slightly edited) of a paper of the same title that appeared in the book Physics and Our World: A Symposium in Honor of Victor F. Weisskopf, published by the American Institute of Physics (1976). The personal tone of the original
- purcell.pdf -> The demonstration involved a tall rectangular transparent vessel of corn syrup, projected by an overhead projector turned on its side. Some essential hand waving could not be reproduced.

This is a talk that I would not, I’m afraid, have the nerve to give under any other circumstances, It’s a story I’ve been saving up to tell Viki. Like so many of you here, I’ve enjoyed from time to time the wonde
- purcell.pdf -> We wander around strictly as amateurs equipped only with some elementary physics, and in the en

## Search the complete library

In [ ]:
mode_used, root_results = dashboard_search(
    paths.osii_root,
    query=ROOT_QUERY,
    mode="lexical",
    top_k=5,
    scope={"scope_type": "root"},
)

print("Mode used:", mode_used)
print("Results:", len(root_results))

## Inspect grounding, not only scores

A score ranks candidates. Object ID, source path, and character offsets make
the candidate defensible. An agent should carry both ranking and grounding
forward when producing an answer or deciding on another tool call.

In [ ]:
for result in root_results:
    print(f"- {result['source_relpath']}  score={result['score']:.3f}")
    print(f"  {result['snippet']}")
    print(f"  chars={result.get('char_start')}:{result.get('char_end')}")

## Reuse retrieval inside a task-specific scope

The search implementation stays the same; only the explicit context changes.
This is why scopes are useful building blocks for future agents.

In [ ]:
collection = next(
    item for item in list_collections(paths.osii_root)
    if item["name"] == "Purcell analysis"
)
collection_scope = {
    "scope_type": "collection",
    "collection_id": collection["id"],
}

COLLECTION_QUERY = "scallop theorem reciprocal motion"

In [ ]:
_, collection_results = dashboard_search(
    paths.osii_root,
    query=COLLECTION_QUERY,
    mode="lexical",
    top_k=5,
    scope=collection_scope,
)

for result in collection_results:
    print(f"- {result['source_relpath']}")
    print(f"  {result['snippet']}")

Lexical retrieval remains available when every optional model endpoint is
offline. That is both a usability guarantee and a research control: later
hybrid results can be compared against a transparent zero-model baseline.